# CSRNet Dense Crowd Counting — Colab Training

Train CSRNet (Improved / Original / MultiScale) on the ShanghaiTech Part A dataset.

**Steps:**
1. Mount Google Drive & navigate to project
2. Install dependencies
3. Verify GPU
4. Generate density maps (first time only)
5. Train
6. Evaluate

## 1. Mount Drive & Set Project Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/LLM Asst. Dense Crowd Counting"
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Files:", os.listdir("."))

## 2. Install Dependencies

In [ ]:
!pip install -q h5py scipy tqdm pillow matplotlib

## 3. Verify GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM    : {props.total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")

## 4. Generate Density Maps (First Time Only)

Skip this cell if `part_A/train_data/ground-truth-h5-s8/` already exists.

In [ ]:
gt_dir = "part_A/train_data/ground-truth-h5-s8"
if os.path.exists(gt_dir) and len(os.listdir(gt_dir)) > 0:
    print(f"Density maps already exist ({len(os.listdir(gt_dir))} files). Skipping.")
else:
    !python preprocessing.py --stride 8 --adaptive --verify

## 5. Train

Adjust parameters below as needed. Default is the Improved CSRNet with CBAM attention.

In [ ]:
!python train.py \
  --model improved \
  --backbone vgg19 \
  --epochs 400 \
  --batch-size 1 \
  --lr 1e-4 \
  --frontend-lr-scale 0.1 \
  --warmup-epochs 10 \
  --optimizer adamw \
  --scheduler cosine \
  --loss combined \
  --ssim-weight 0.01 \
  --count-weight 0.01 \
  --crop-size 512 \
  --num-workers 2 \
  --seed 42

### 5b. Resume Training from Checkpoint

Use this cell if Colab disconnected or you want to continue training.

In [ ]:
# Uncomment to resume:
# !python train.py \
#   --resume checkpoints/best_model.pth \
#   --model improved \
#   --backbone vgg19 \
#   --epochs 400 \
#   --batch-size 1 \
#   --lr 1e-4 \
#   --frontend-lr-scale 0.1 \
#   --warmup-epochs 10 \
#   --optimizer adamw \
#   --scheduler cosine \
#   --loss combined \
#   --count-weight 0.01 \
#   --crop-size 512 \
#   --num-workers 2 \
#   --seed 42

## 6. Evaluate on Test Set

In [ ]:
from model import create_model
from dataset import ShanghaiTechDatasetImproved
from train import validate, validate_tta
from torch.utils.data import DataLoader
import torch, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Find the best checkpoint
ckpt_path = "checkpoints/best_model.pth"
if not os.path.exists(ckpt_path):
    ckpt_path = "best_model.pth"

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt.get('config', {})

model = create_model(
    cfg.get('MODEL_TYPE', 'improved'),
    load_weights=False,
    use_attention=cfg.get('USE_ATTENTION', True),
    use_bn=cfg.get('USE_BN', False),
    dropout_rate=cfg.get('DROPOUT_RATE', 0.0),
).to(device)
model.load_state_dict(ckpt['model_state_dict'])

print(f"Loaded: epoch={ckpt.get('epoch','?')}, best_mae={ckpt.get('best_mae','?')}")

# Standard validation
val_ds = ShanghaiTechDatasetImproved(
    "./part_A/test_data", gt_folder="ground-truth-h5-s8", training=False,
)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=2)
mae, mse = validate(model, val_loader, device, use_amp=True)
print(f"\nSingle-pass — MAE: {mae:.2f}  |  RMSE: {mse:.2f}")

# TTA validation (multi-scale + flip)
tta_mae, tta_mse = validate_tta(model, "./part_A/test_data", device, use_amp=True)
print(f"TTA         — MAE: {tta_mae:.2f}  |  RMSE: {tta_mse:.2f}")

## 7. Visualize a Prediction

In [ ]:
import glob
import random
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

# Pick a random test image
test_imgs = sorted(glob.glob("part_A/test_data/images/*.jpg"))
img_path = random.choice(test_imgs)
img = Image.open(img_path).convert("RGB")

# Stride-align
w, h = img.size
img = img.crop((0, 0, (w // 8) * 8, (h // 8) * 8))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model.eval()
with torch.no_grad():
    x = transform(img).unsqueeze(0).to(device)
    if device.type == 'cuda':
        with torch.amp.autocast(device_type='cuda'):
            output = model(x)
    else:
        output = model(x)

dm = output.squeeze().cpu().numpy()
pred_count = int(round(dm.sum()))

# Load GT if available
import h5py
gt_path = img_path.replace("images", "ground-truth-h5-s8").replace(".jpg", ".h5")
gt_count = None
if os.path.exists(gt_path):
    with h5py.File(gt_path, 'r') as f:
        gt_count = int(round(float(f['density'][:].sum())))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img)
axes[0].set_title(f"Original — GT: {gt_count or 'N/A'}")
axes[0].axis('off')

axes[1].imshow(dm, cmap='jet')
axes[1].set_title(f"Density Map — Pred: {pred_count}")
axes[1].axis('off')

axes[2].imshow(img)
dm_resized = np.array(Image.fromarray(dm.astype(np.float32)).resize(img.size, Image.BILINEAR))
axes[2].imshow(dm_resized, cmap='jet', alpha=0.5)
axes[2].set_title("Overlay")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"\nImage: {os.path.basename(img_path)}")
print(f"Predicted: {pred_count}")
if gt_count is not None:
    print(f"Ground Truth: {gt_count}")
    print(f"Error: {abs(pred_count - gt_count)}")

## 8. Save Model to Drive

In [ ]:
import shutil
src = "checkpoints/best_model.pth"
dst = "/content/drive/MyDrive/best_model_colab.pth"
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f"Copied {src} -> {dst}")
else:
    print(f"{src} not found. Check the checkpoints directory.")